In [1]:
import subprocess
import os
import sys
import re
import numpy as np
from scipy.optimize import minimize
from scipy.optimize import basinhopping

In [2]:
model=['M2_1.cfg','M4.cfg','S2_1.cfg','T3.cfg']

In [3]:
bindir='./bin'
data_path='./data'
output_directory='./'

bindir=os.path.realpath(bindir)
data_path = os.path.realpath(data_path)

if not os.path.isdir(output_directory):
    os.makedirs(output_directory)

In [18]:
def getEQ(x_,model=model):
    EQ=' " '
    for i in range(len(x_)):
        if len(EQ)>0:
            EQ=EQ+' + '
        EQ=EQ+'( '+"%.9f" % x_[i]+' * '+model[i]+' ) '
    EQ=EQ+'"'
    return EQ

def loss(seqfile,x_=[1,1,1,1]):
    
    EQ=getEQ(x_,model)
    PSOLVE=os.path.join(bindir,'psolve')+' -D "" -o '+\
                                os.path.join(output_directory,'machine.cfg')+' -x '+EQ
    LLK=os.path.join(bindir,'llk')+' -f '+os.path.join(output_directory,'machine.cfg')+\
                                    ' -T symbolic -s '+os.path.join(data_path,seqfile)
    ret=subprocess.check_output(PSOLVE, shell=True)
    ret=subprocess.check_output(LLK, shell=True)
    return float(re.findall("\d+\.\d+", ret)[0])

def lsmash(seq1,seq2,datapath='./',x_=[1,1,1,1]):
    seq1=os.path.join(data_path,seq1)
    seq2=os.path.join(data_path,seq2)
    
    return np.linalg.norm(np.array(loss(seq1,x_=x_))-np.array(loss(seq2,x_=x_)))

In [139]:
x_=[1,1,1,-0]
print lsmash('T3.dat','M2_1.dat',datapath=data_path,x_=x_)
print lsmash('T3_2.dat','M2_1.dat',datapath=data_path,x_=x_)
print lsmash('S2_2.dat','M2_1.dat',datapath=data_path,x_=x_)
print lsmash('M4.dat','M2.dat',datapath=data_path,x_=x_)
print lsmash('T3_2.dat','T3.dat',datapath=data_path,x_=x_)
print "--"
print lsmash('S2_3.dat','S2_1.dat',datapath=data_path,x_=x_)
print lsmash('S2_2.dat','S2_1.dat',datapath=data_path,x_=x_)
print lsmash('S2_2.dat','S2_3.dat',datapath=data_path,x_=x_)
print lsmash('M2.dat','M2_1.dat',datapath=data_path,x_=x_)
print lsmash('T3_2.dat','T3_1.dat',datapath=data_path,x_=x_)


0.32383000000000006
0.14398
1.04577
0.4539599999999999
0.17985000000000007
--
0.14751000000000003
0.015349999999999975
0.13216000000000006
0.005210000000000381
0.005519999999999747


In [48]:
!cat machine.cfg

%PITILDE: size(5)
#PITILDE 
0.00526316 0.994737 
0.0181818 0.981818 
0.7 0.3 
0.972 0.028 
0.991837 0.00816327 
%CONNX: size(5)
#CONNX 
1 4 
2 2 
0 3 
4 0 
2 1 
